# Interactive HVDC LCC V–I Control Characteristic

This notebook is a **didactic static characteristic** for understanding the figure. It is not a time-domain simulation and it is not a complete HVDC load-flow model.

The plot uses the **positive magnitude** of DC voltage.

Main relations:

$$
V_{dR}=V_{d0R}\cos(\alpha_{\min})-\frac{3X_{cR}}{\pi}I_d
$$

$$
V_{dI}=V_{d0I}\cos(\gamma_{\min})-\frac{3X_{cI}}{\pi}I_d
$$

$$
I_{\mathrm{order},I}=I_{\mathrm{order},R}-\Delta I_d
$$

$$
V_{dc}=V_{dc,\mathrm{order}}
$$

In normal operation in this simplified control map, the **rectifier controls DC current** and the **inverter controls DC voltage**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [3]:
# Build the interactive model.
Vd0R = widgets.FloatSlider(value=500, min=350, max=650, step=5, description='Vd0R [kV]')
alpha_min = widgets.FloatSlider(value=5, min=0, max=25, step=0.5, description='alpha min [deg]')
XcR = widgets.FloatSlider(value=12, min=1, max=30, step=0.5, description='XcR [ohm]')

I_order_R = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.05, description='Iorder,R [kA]')
delta_I = widgets.FloatSlider(value=0.10, min=0.02, max=0.50, step=0.01, description='Delta I [kA]')

Vd0I = widgets.FloatSlider(value=500, min=350, max=650, step=5, description='Vd0I [kV]')
gamma_min = widgets.FloatSlider(value=18, min=5, max=35, step=0.5, description='gamma min [deg]')
XcI = widgets.FloatSlider(value=12, min=1, max=30, step=0.5, description='XcI [ohm]')
Vdc_order = widgets.FloatSlider(value=450, min=300, max=600, step=5, description='Vdc order [kV]')

Imax = widgets.FloatSlider(value=2.8, min=1.0, max=5.0, step=0.1, description='Imax [kA]')

output = widgets.Output()

controls_left = widgets.VBox([Vd0R, alpha_min, XcR, I_order_R, delta_I])
controls_right = widgets.VBox([Vd0I, gamma_min, XcI, Vdc_order, Imax])
controls = widgets.HBox([controls_left, controls_right])

def update_plot(change=None):
    with output:
        clear_output(wait=True)

        rc_r = 3 * XcR.value / np.pi
        rc_i = 3 * XcI.value / np.pi
        i_order_i = I_order_R.value - delta_I.value

        i = np.linspace(0, Imax.value, 600)

        v_alpha = Vd0R.value * np.cos(np.deg2rad(alpha_min.value)) - rc_r * i
        v_gamma = Vd0I.value * np.cos(np.deg2rad(gamma_min.value)) - rc_i * i

        if rc_i > 0:
            i_gamma_cross = (
                Vd0I.value * np.cos(np.deg2rad(gamma_min.value)) - Vdc_order.value
            ) / rc_i
        else:
            i_gamma_cross = np.nan

        v_alpha_order = (
            Vd0R.value * np.cos(np.deg2rad(alpha_min.value))
            - rc_r * I_order_R.value
        )

        fig, ax = plt.subplots(figsize=(12, 6.2))

        mask_alpha = i <= I_order_R.value
        ax.plot(
            i[mask_alpha],
            v_alpha[mask_alpha],
            linewidth=2.2,
            label=r'Rectifier limit: $\alpha=\alpha_{min}$'
        )

        if np.isfinite(i_gamma_cross):
            mask_gamma = i >= max(0, i_gamma_cross)
            ax.plot(
                i[mask_gamma],
                v_gamma[mask_gamma],
                linewidth=2.2,
                label=r'Inverter limit: $\gamma=\gamma_{min}$'
            )

            x1 = max(0, i_order_i)
            x2 = min(Imax.value, i_gamma_cross)
            if x2 >= x1:
                ax.hlines(
                    Vdc_order.value,
                    x1,
                    x2,
                    linewidth=3.0,
                    label=r'Inverter voltage control: $V_{dc}=V_{dc,order}$'
                )

        ax.vlines(
            I_order_R.value,
            0,
            max(0, v_alpha_order),
            linewidth=2.5,
            linestyles='-',
            label=r'Rectifier current control: $I_d=I_{order,R}$'
        )

        ax.vlines(
            i_order_i,
            0,
            Vdc_order.value,
            linewidth=2.0,
            linestyles='--',
            label=r'Inverter backup current control'
        )

        margin_y = 0.25 * Vdc_order.value
        ax.annotate(
            '',
            xy=(I_order_R.value, margin_y),
            xytext=(i_order_i, margin_y),
            arrowprops=dict(arrowstyle='<->', linewidth=1.8)
        )
        ax.text(
            0.5 * (I_order_R.value + i_order_i),
            margin_y * 1.05,
            r'$\Delta I_d$',
            ha='center',
            va='bottom',
            fontsize=12
        )

        ax.plot(
            I_order_R.value,
            Vdc_order.value,
            marker='o',
            markersize=8,
            linestyle='None',
            label='Normal operating point'
        )

        ax.annotate(
            'Normal operating point',
            xy=(I_order_R.value, Vdc_order.value),
            xytext=(12, -32),
            textcoords='offset points',
            arrowprops=dict(arrowstyle='->')
        )

        ax.set_xlabel(r'DC current, $I_d$ [kA]')
        ax.set_ylabel(r'DC voltage magnitude, $V_{dc}$ [kV]')
        ax.set_title('HVDC LCC Static Control Characteristic')
        ax.grid(True, alpha=0.25)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        ax.legend(loc='best', fontsize=9)
        plt.show()

        status = []
        status.append(
            f"**Inverter current order:** "
            f"$I_{{order,I}}={i_order_i:.3f}\,\mathrm{{kA}}$"
        )
        status.append(
            f"**Rectifier commutation slope:** "
            f"$3X_{{cR}}/\\pi={rc_r:.3f}\,\mathrm{{kV/kA}}$"
        )
        status.append(
            f"**Inverter commutation slope:** "
            f"$3X_{{cI}}/\\pi={rc_i:.3f}\,\mathrm{{kV/kA}}$"
        )
        status.append(
            f"**Rectifier $\\alpha_{{min}}$ voltage at current order:** "
            f"$V_{{dR}}={v_alpha_order:.1f}\,\mathrm{{kV}}$"
        )

        if np.isfinite(i_gamma_cross):
            status.append(
                f"**Intersection of $V_{{dc,order}}$ with the "
                f"$\\gamma_{{min}}$ characteristic:** "
                f"$I_d={i_gamma_cross:.3f}\,\mathrm{{kA}}$"
            )

        if i_order_i < 0:
            status.append("⚠️ The selected current margin is larger than the rectifier current order.")
        if np.isfinite(i_gamma_cross) and i_gamma_cross < I_order_R.value:
            status.append(
                "⚠️ With these settings, the gamma-limit intersection lies to the left of the "
                "rectifier current order. This is useful for exploring controller-limit conflicts."
            )

        display(Markdown("\n\n".join(status)))

for widget in [
    Vd0R, alpha_min, XcR, I_order_R, delta_I,
    Vd0I, gamma_min, XcI, Vdc_order, Imax
]:
    widget.observe(update_plot, names='value')

display(controls, output)
update_plot()

<>:140: SyntaxWarning: invalid escape sequence '\,'
<>:144: SyntaxWarning: invalid escape sequence '\,'
<>:148: SyntaxWarning: invalid escape sequence '\,'
<>:152: SyntaxWarning: invalid escape sequence '\,'
<>:159: SyntaxWarning: invalid escape sequence '\,'
<>:140: SyntaxWarning: invalid escape sequence '\,'
<>:144: SyntaxWarning: invalid escape sequence '\,'
<>:148: SyntaxWarning: invalid escape sequence '\,'
<>:152: SyntaxWarning: invalid escape sequence '\,'
<>:159: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_4615/642056803.py:140: SyntaxWarning: invalid escape sequence '\,'
  f"$I_{{order,I}}={i_order_i:.3f}\,\mathrm{{kA}}$"
/tmp/ipykernel_4615/642056803.py:144: SyntaxWarning: invalid escape sequence '\,'
  f"$3X_{{cR}}/\\pi={rc_r:.3f}\,\mathrm{{kV/kA}}$"
/tmp/ipykernel_4615/642056803.py:148: SyntaxWarning: invalid escape sequence '\,'
  f"$3X_{{cI}}/\\pi={rc_i:.3f}\,\mathrm{{kV/kA}}$"
/tmp/ipykernel_4615/642056803.py:152: SyntaxWarning: invalid escape sequence '\,

Output()

## Suggested experiments

1. Increase **alpha min** and observe the rectifier limit move downward.
2. Increase **XcR** and observe the rectifier limit become more steeply descending.
3. Increase **Iorder,R** and observe the rectifier current-control line move to the right.
4. Increase **Delta I** and observe the inverter backup-current line move to the left.
5. Increase **Vdc order** and observe the horizontal inverter-control segment move upward.
6. Increase **gamma min** and observe the inverter gamma-limit curve move downward.
7. Increase **XcI** and observe the inverter gamma-limit curve become more steeply descending.

The most important geometric point is that the inverter backup current order is lower than the rectifier current order:

\[
I_{\mathrm{order},I}=I_{\mathrm{order},R}-\Delta I_d
\]

Therefore, the inverter current-control line should appear **to the left** of the rectifier current-control line.